# LeetCode 1594 — Maximum Non Negative Product in a Matrix bằng Python

Link đề: https://leetcode.com/problems/maximum-non-negative-product-in-a-matrix/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Matrix

Skeleton LeetCode:

```python
class Solution:
    def maxProductPath(self, grid: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đếm và liệt kê mọi đường đi phải/xuống trong lưới để có lời giải **vét cạn** làm chuẩn.
2. Hiểu vì sao bài **tích** khác bài **tổng**: nhân với số âm làm **lớn nhất ↔ nhỏ nhất đổi chỗ**, nên DP phải giữ **cả (max, min)** ở mỗi ô.
3. Viết DP bảng 2 chiều, rồi tối ưu xuống **1 chiều** (mảng lăn) và viết bản **đệ quy có nhớ**.
4. Hiểu tại sao **chỉ lấy modulo ở cuối**: modulo phá thứ tự so sánh; và xử lý trả `-1` khi tích lớn nhất âm (lưu ý `0` vẫn là không âm).
5. Kiểm thử ngẫu nhiên với seed cố định, so sánh DP với vét cạn.

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `grid` kích thước `m × n`. Bắt đầu ở góc trên trái `(0, 0)`, mỗi bước chỉ được đi **sang phải** hoặc **xuống dưới**. Xét mọi đường đi từ `(0, 0)` tới góc dưới phải `(m−1, n−1)`; **tích** của một đường đi là tích mọi số trên các ô đi qua.

Tìm tích **không âm lớn nhất** trong các đường đi, trả về nó **modulo `10⁹ + 7`**. Nếu tích lớn nhất **âm** thì trả `-1`. Lưu ý: phép modulo thực hiện **sau khi** đã tìm được tích lớn nhất.

**Ví dụ 1:** `grid = [[-1,-2,-3],[-2,-3,-3],[-3,-3,-2]]` → `-1` (không có đường nào cho tích không âm).

**Ví dụ 2:** `grid = [[1,-2,1],[1,-2,1],[3,-4,1]]` → `8` (đường `1 · 1 · (−2) · (−4) · 1 = 8`).

**Ví dụ 3:** `grid = [[1,3],[0,-4]]` → `0` (đường `1 · 0 · (−4) = 0`).

**Ràng buộc:**
- `m == grid.length`, `n == grid[i].length`
- `1 ≤ m, n ≤ 15`
- `−4 ≤ grid[i][j] ≤ 4`

In [1]:
import random
import time
from math import comb
from functools import lru_cache
from typing import List

MOD = 10**9 + 7
examples = [
    ([[-1, -2, -3], [-2, -3, -3], [-3, -3, -2]], -1),
    ([[1, -2, 1], [1, -2, 1], [3, -4, 1]], 8),
    ([[1, 3], [0, -4]], 0),
]
for g, want in examples:
    print(f"{len(g)}x{len(g[0])} grid={g} -> mong đợi {want}")

3x3 grid=[[-1, -2, -3], [-2, -3, -3], [-3, -3, -2]] -> mong đợi -1
3x3 grid=[[1, -2, 1], [1, -2, 1], [3, -4, 1]] -> mong đợi 8
2x2 grid=[[1, 3], [0, -4]] -> mong đợi 0


## 3. Kiến thức nền

### 3.1 Lưới 2 chiều trong Python

`grid[i][j]` là hàng `i`, cột `j`. Tạo bảng `m × n` đúng cách: `[[0] * n for _ in range(m)]` (không dùng `[[0] * n] * m` — các hàng sẽ là **cùng một list**).

In [2]:
m, n = 2, 3
ok = [[0] * n for _ in range(m)]
bad = [[0] * n] * m
ok[0][0] = 9; bad[0][0] = 9
print("đúng:", ok)
print("sai :", bad, "<- cả hai hàng đều đổi!")

đúng: [[9, 0, 0], [0, 0, 0]]
sai : [[9, 0, 0], [9, 0, 0]] <- cả hai hàng đều đổi!


### 3.2 Số đường đi phải/xuống

Mỗi đường đi gồm `m − 1` bước xuống và `n − 1` bước phải theo thứ tự nào đó ⇒ số đường đi là `C(m+n−2, m−1)`. Với `15 × 15`, con số này là hơn 40 triệu — vét cạn không khả thi ở kích thước tối đa.

In [3]:
for m, n in [(3, 3), (5, 5), (10, 10), (15, 15)]:
    print(f"{m}x{n}: {comb(m + n - 2, m - 1):,} đường đi")

3x3: 6 đường đi
5x5: 70 đường đi
10x10: 48,620 đường đi
15x15: 40,116,600 đường đi


### 3.3 Nhân với số âm đảo thứ tự

Nếu `a ≤ b` và `x < 0` thì `a·x ≥ b·x`. Vì vậy giá trị **lớn nhất** sau khi nhân với số âm đến từ giá trị **nhỏ nhất** trước đó. Đây là lý do phải theo dõi cả min.

In [4]:
vals = [-6, -1, 0, 5, 8]
x = -2
print("trước:", vals, " max =", max(vals), " min =", min(vals))
prod = [v * x for v in vals]
print("sau ×(-2):", prod, " max =", max(prod), "(đến từ min cũ", min(vals), ")")

trước: [-6, -1, 0, 5, 8]  max = 8  min = -6
sau ×(-2): [12, 2, 0, -10, -16]  max = 12 (đến từ min cũ -6 )


### 3.4 Modulo phá thứ tự

`a > b` **không** suy ra `a % M > b % M`. Ví dụ `M = 10`: `12 > 9` nhưng `12 % 10 = 2 < 9`. Nên không được lấy modulo trước khi so sánh để chọn max. Python có số nguyên lớn tùy ý nên ta giữ giá trị **thật** suốt DP (giá trị tối đa `4^29 ≈ 2.9·10¹⁷`, vẫn vừa `int64` ở ngôn ngữ khác).

Thêm nữa: trong Python `(-5) % 7 == 2` — số âm modulo ra số **dương**, nên phải kiểm tra dấu **trước** khi lấy modulo.

In [5]:
M = 10
print(12 > 9, 12 % M > 9 % M)
print("(-5) % 7 =", (-5) % 7, "<- Python trả số dương!")
print("4**29 =", 4**29, " < 2**63 ?", 4**29 < 2**63)

True False
(-5) % 7 = 2 <- Python trả số dương!
4**29 = 288230376151711744  < 2**63 ? True


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 3 `[[1,3],[0,-4]]` — chỉ có 2 đường:
- phải rồi xuống: `1 · 3 · (−4) = −12`
- xuống rồi phải: `1 · 0 · (−4) = 0`

Max = 0 ≥ 0 ⇒ trả 0.

Với ví dụ 2, ta liệt kê cả 6 đường bằng code bên dưới. Quan sát quan trọng: đường tốt nhất đi qua **hai** số âm `−2` và `−4` để tích thành dương. Nếu tại ô `(1,1)` (giá trị `−2`) ta chỉ nhớ tích **lớn nhất** tới đó, ta có thể đánh mất tích **âm nhất** — thứ sẽ trở thành lớn nhất khi gặp `−4`.

In [6]:
def all_paths(grid):
    m, n = len(grid), len(grid[0])
    res = []
    def go(i, j, path, prod):
        path = path + [grid[i][j]]; prod *= grid[i][j]
        if (i, j) == (m - 1, n - 1):
            res.append((path, prod)); return
        if i + 1 < m: go(i + 1, j, path, prod)
        if j + 1 < n: go(i, j + 1, path, prod)
    go(0, 0, [], 1)
    return res

for path, prod in all_paths(examples[1][0]):
    print(f"{path} -> tích {prod}")

[1, 1, 3, -4, 1] -> tích -12
[1, 1, -2, -4, 1] -> tích 8
[1, 1, -2, 1, 1] -> tích -2
[1, -2, -2, -4, 1] -> tích -16
[1, -2, -2, 1, 1] -> tích 4
[1, -2, 1, 1, 1] -> tích -2


## 5. Cách 1 — Brute Force / baseline: DFS mọi đường đi

**Ý tưởng:** đệ quy từ `(0,0)`, mang theo tích hiện tại; tới đích thì cập nhật max. Cuối cùng: max âm ⇒ `-1`, ngược lại `max % MOD`.

In [7]:
def max_product_brute(grid):
    m, n = len(grid), len(grid[0])
    best = None
    def dfs(i, j, prod):
        nonlocal best
        prod *= grid[i][j]
        if i == m - 1 and j == n - 1:
            if best is None or prod > best:
                best = prod
            return
        if i + 1 < m: dfs(i + 1, j, prod)
        if j + 1 < n: dfs(i, j + 1, prod)
    dfs(0, 0, 1)
    return -1 if best < 0 else best % MOD

for g, want in examples:
    print(max_product_brute(g), "| mong đợi", want)

-1 | mong đợi -1
8 | mong đợi 8
0 | mong đợi 0


**Độ phức tạp:** `C(m+n−2, m−1)` đường, mỗi đường dài `m+n−1` ⇒ thời gian hàm mũ (khoảng `O(2^(m+n))`), bộ nhớ O(m+n) cho đệ quy.

**Nút thắt:** số đường tăng bùng nổ. Đo thực tế:

In [8]:
random.seed(0)
for k in [6, 8, 10]:
    g = [[random.randint(-4, 4) or 1 for _ in range(k)] for _ in range(k)]
    t = time.perf_counter(); max_product_brute(g)
    print(f"{k}x{k}: {comb(2 * k - 2, k - 1):6d} đường, {time.perf_counter() - t:.3f}s")

6x6:    252 đường, 0.000s
8x8:   3432 đường, 0.002s
10x10:  48620 đường, 0.022s


## 6. Ý tưởng thuật toán / bước đột phá

Ở bài **tổng lớn nhất** (LC 64 kiểu max), ta chỉ cần `best[i][j] = grid[i][j] + max(best[trên], best[trái])`. Ở bài **tích**, điều đó sai vì dấu.

**Insight:** với mọi ô `(i, j)`, hãy nhớ **hai** số:
- `hi[i][j]` = tích **lớn nhất** của một đường từ `(0,0)` tới `(i,j)`,
- `lo[i][j]` = tích **nhỏ nhất** của một đường từ `(0,0)` tới `(i,j)`.

Mọi đường tới `(i,j)` đi qua ô trên hoặc ô trái, rồi nhân với `v = grid[i][j]`. Khi nhân với `v`:
- Nếu `v ≥ 0`: max mới đến từ max cũ, min mới từ min cũ.
- Nếu `v < 0`: max mới đến từ **min** cũ, min mới từ **max** cũ.

Cách viết gọn không cần chia trường hợp: lấy 4 ứng viên `{hi_trên·v, lo_trên·v, hi_trái·v, lo_trái·v}`, `hi = max`, `lo = min`.

**Invariant:** `hi[i][j]` và `lo[i][j]` đúng là max/min trên **mọi** đường tới `(i,j)` (vì hàm `x ↦ x·v` đơn điệu, cực trị của tập ảnh đạt tại cực trị của tập gốc).

Cuối cùng: `hi[m−1][n−1] < 0` ⇒ `-1`; ngược lại `hi % MOD`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP bảng 2 chiều (max, min)

Từng đoạn:
1. `hi[0][0] = lo[0][0] = grid[0][0]`.
2. Hàng đầu / cột đầu chỉ có một hướng tới ⇒ tích dồn (max = min).
3. Ô bên trong: 4 ứng viên như mục 6.
4. Kết quả kiểm dấu rồi mới modulo.

In [9]:
def max_product_dp2d(grid):
    m, n = len(grid), len(grid[0])
    hi = [[0] * n for _ in range(m)]
    lo = [[0] * n for _ in range(m)]
    hi[0][0] = lo[0][0] = grid[0][0]
    for j in range(1, n):                       # hàng đầu
        hi[0][j] = lo[0][j] = hi[0][j - 1] * grid[0][j]
    for i in range(1, m):                       # cột đầu
        hi[i][0] = lo[i][0] = hi[i - 1][0] * grid[i][0]
    for i in range(1, m):
        for j in range(1, n):
            v = grid[i][j]
            cand = (hi[i - 1][j] * v, lo[i - 1][j] * v, hi[i][j - 1] * v, lo[i][j - 1] * v)
            hi[i][j] = max(cand)
            lo[i][j] = min(cand)
    best = hi[m - 1][n - 1]
    return -1 if best < 0 else best % MOD

for g, want in examples:
    print(max_product_dp2d(g), "| mong đợi", want)

-1 | mong đợi -1
8 | mong đợi 8
0 | mong đợi 0


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def maxProductPath(self, grid: List[List[int]]) -> int:
        MOD = 10**9 + 7
        m, n = len(grid), len(grid[0])
        hi = [[0] * n for _ in range(m)]
        lo = [[0] * n for _ in range(m)]
        hi[0][0] = lo[0][0] = grid[0][0]
        for j in range(1, n):
            hi[0][j] = lo[0][j] = hi[0][j - 1] * grid[0][j]
        for i in range(1, m):
            hi[i][0] = lo[i][0] = hi[i - 1][0] * grid[i][0]
        for i in range(1, m):
            for j in range(1, n):
                v = grid[i][j]
                c = (hi[i-1][j] * v, lo[i-1][j] * v, hi[i][j-1] * v, lo[i][j-1] * v)
                hi[i][j], lo[i][j] = max(c), min(c)
        best = hi[-1][-1]
        return -1 if best < 0 else best % MOD
```

**Độ phức tạp:** Time **O(m·n)**, Space **O(m·n)**.

### 7.2 Cách 3 — DP mảng lăn 1 chiều

Ô `(i, j)` chỉ cần hàng trên (`i−1`) và ô trái cùng hàng. Dùng hai mảng độ dài `n`: trước khi ghi đè, `hi[j]` vẫn là giá trị **hàng trên**, còn `hi[j−1]` đã là giá trị **hàng hiện tại** (ô trái). Bộ nhớ giảm xuống O(n).

In [10]:
def max_product_dp1d(grid):
    m, n = len(grid), len(grid[0])
    hi = [0] * n
    lo = [0] * n
    for i in range(m):
        for j in range(n):
            v = grid[i][j]
            if i == 0 and j == 0:
                hi[0] = lo[0] = v
                continue
            cand = []
            if i > 0:                      # hi[j], lo[j] lúc này là của hàng trên
                cand += [hi[j] * v, lo[j] * v]
            if j > 0:                      # hi[j-1], lo[j-1] đã là của hàng hiện tại
                cand += [hi[j - 1] * v, lo[j - 1] * v]
            hi[j], lo[j] = max(cand), min(cand)
    return -1 if hi[-1] < 0 else hi[-1] % MOD

for g, want in examples:
    print(max_product_dp1d(g), "| mong đợi", want)

-1 | mong đợi -1
8 | mong đợi 8
0 | mong đợi 0


Code nộp LeetCode của Cách 3:

```python
class Solution:
    def maxProductPath(self, grid: List[List[int]]) -> int:
        MOD = 10**9 + 7
        m, n = len(grid), len(grid[0])
        hi, lo = [0] * n, [0] * n
        for i in range(m):
            for j in range(n):
                v = grid[i][j]
                if i == 0 and j == 0:
                    hi[0] = lo[0] = v
                    continue
                c = []
                if i > 0:
                    c += [hi[j] * v, lo[j] * v]
                if j > 0:
                    c += [hi[j - 1] * v, lo[j - 1] * v]
                hi[j], lo[j] = max(c), min(c)
        return -1 if hi[-1] < 0 else hi[-1] % MOD
```

**Độ phức tạp:** Time O(m·n), Space **O(n)**.

### 7.3 Cách 4 — Đệ quy có nhớ (top-down)

`f(i, j)` trả về cặp `(max, min)` tích từ `(0,0)` tới `(i,j)`. Công thức giống hệt, chỉ là viết theo kiểu "hỏi ô trên và ô trái". `lru_cache` đảm bảo mỗi ô tính một lần. Độ sâu đệ quy tối đa `m + n − 1 ≤ 29` — an toàn.

In [11]:
def max_product_memo(grid):
    m, n = len(grid), len(grid[0])

    @lru_cache(maxsize=None)
    def f(i, j):
        v = grid[i][j]
        if i == 0 and j == 0:
            return v, v
        cand = []
        if i > 0:
            a, b = f(i - 1, j); cand += [a * v, b * v]
        if j > 0:
            a, b = f(i, j - 1); cand += [a * v, b * v]
        return max(cand), min(cand)

    best = f(m - 1, n - 1)[0]
    return -1 if best < 0 else best % MOD

for g, want in examples:
    print(max_product_memo(g), "| mong đợi", want)

-1 | mong đợi -1
8 | mong đợi 8
0 | mong đợi 0


Code nộp LeetCode của Cách 4:

```python
from functools import lru_cache

class Solution:
    def maxProductPath(self, grid: List[List[int]]) -> int:
        MOD = 10**9 + 7
        m, n = len(grid), len(grid[0])
        @lru_cache(maxsize=None)
        def f(i, j):
            v = grid[i][j]
            if i == 0 and j == 0:
                return v, v
            c = []
            if i > 0:
                a, b = f(i - 1, j); c += [a * v, b * v]
            if j > 0:
                a, b = f(i, j - 1); c += [a * v, b * v]
            return max(c), min(c)
        best = f(m - 1, n - 1)[0]
        return -1 if best < 0 else best % MOD
```

**Độ phức tạp:** Time O(m·n), Space O(m·n) cho cache + O(m+n) ngăn xếp.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2 `[[1,-2,1],[1,-2,1],[3,-4,1]]`. Mỗi ô ghi `(hi, lo)`:

| | cột 0 | cột 1 | cột 2 |
|---|---|---|---|
| hàng 0 | (1, 1) | (−2, −2) | (−2, −2) |
| hàng 1 | (1, 1) | (4, −2) | (4, −2) |
| hàng 2 | (3, 3) | (8, −16) | (8, −16) |

Ô `(1,1)`, `v = −2`: ứng viên từ trên `(−2)·(−2) = 4`, từ trái `1·(−2) = −2` ⇒ `(4, −2)`. Ô `(2,1)`, `v = −4`: từ trên `(4, −2)` ⇒ `−16, 8`; từ trái `(3,3)` ⇒ `−12` ⇒ `(8, −16)`. Kết quả `8`.

In [12]:
def trace_dp(grid):
    m, n = len(grid), len(grid[0])
    hi = [[None] * n for _ in range(m)]
    lo = [[None] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            v = grid[i][j]
            if i == 0 and j == 0:
                hi[i][j] = lo[i][j] = v
                print(f"({i},{j}) v={v:2d}: khởi tạo -> hi={v}, lo={v}")
                continue
            cand = []
            src = []
            if i > 0:
                cand += [hi[i-1][j] * v, lo[i-1][j] * v]; src.append(f"trên({hi[i-1][j]},{lo[i-1][j]})")
            if j > 0:
                cand += [hi[i][j-1] * v, lo[i][j-1] * v]; src.append(f"trái({hi[i][j-1]},{lo[i][j-1]})")
            hi[i][j], lo[i][j] = max(cand), min(cand)
            print(f"({i},{j}) v={v:2d}: {' '.join(src):24s} ứng viên {cand} -> hi={hi[i][j]}, lo={lo[i][j]}")
    best = hi[-1][-1]
    res = -1 if best < 0 else best % MOD
    print("bảng (hi, lo):")
    for i in range(m):
        print("  ", [(hi[i][j], lo[i][j]) for j in range(n)])
    print("=> kết quả", res)
    return res

trace_dp(examples[1][0])

(0,0) v= 1: khởi tạo -> hi=1, lo=1
(0,1) v=-2: trái(1,1)                ứng viên [-2, -2] -> hi=-2, lo=-2
(0,2) v= 1: trái(-2,-2)              ứng viên [-2, -2] -> hi=-2, lo=-2
(1,0) v= 1: trên(1,1)                ứng viên [1, 1] -> hi=1, lo=1
(1,1) v=-2: trên(-2,-2) trái(1,1)    ứng viên [4, 4, -2, -2] -> hi=4, lo=-2
(1,2) v= 1: trên(-2,-2) trái(4,-2)   ứng viên [-2, -2, 4, -2] -> hi=4, lo=-2
(2,0) v= 3: trên(1,1)                ứng viên [3, 3] -> hi=3, lo=3
(2,1) v=-4: trên(4,-2) trái(3,3)     ứng viên [-16, 8, -12, -12] -> hi=8, lo=-16
(2,2) v= 1: trên(4,-2) trái(8,-16)   ứng viên [4, -2, 8, -16] -> hi=8, lo=-16
bảng (hi, lo):
   [(1, 1), (-2, -2), (-2, -2)]
   [(1, 1), (4, -2), (4, -2)]
   [(3, 3), (8, -16), (8, -16)]
=> kết quả 8


8

In [13]:
trace_dp(examples[0][0])

(0,0) v=-1: khởi tạo -> hi=-1, lo=-1
(0,1) v=-2: trái(-1,-1)              ứng viên [2, 2] -> hi=2, lo=2
(0,2) v=-3: trái(2,2)                ứng viên [-6, -6] -> hi=-6, lo=-6
(1,0) v=-2: trên(-1,-1)              ứng viên [2, 2] -> hi=2, lo=2
(1,1) v=-3: trên(2,2) trái(2,2)      ứng viên [-6, -6, -6, -6] -> hi=-6, lo=-6
(1,2) v=-3: trên(-6,-6) trái(-6,-6)  ứng viên [18, 18, 18, 18] -> hi=18, lo=18
(2,0) v=-3: trên(2,2)                ứng viên [-6, -6] -> hi=-6, lo=-6
(2,1) v=-3: trên(-6,-6) trái(-6,-6)  ứng viên [18, 18, 18, 18] -> hi=18, lo=18
(2,2) v=-2: trên(18,18) trái(18,18)  ứng viên [-36, -36, -36, -36] -> hi=-36, lo=-36
bảng (hi, lo):
   [(-1, -1), (2, 2), (-6, -6)]
   [(2, 2), (-6, -6), (18, 18)]
   [(-6, -6), (18, 18), (-36, -36)]
=> kết quả -1


-1

Ở ví dụ 1, `hi` của ô đích là âm (mọi đường đi qua 5 số âm ⇒ tích luôn âm) ⇒ `-1`.

## 9. Code nộp LeetCode

```python
from typing import List

class Solution:
    def maxProductPath(self, grid: List[List[int]]) -> int:
        MOD = 10**9 + 7
        m, n = len(grid), len(grid[0])
        hi, lo = [0] * n, [0] * n          # max / min tích tới mỗi cột của hàng hiện tại
        for i in range(m):
            for j in range(n):
                v = grid[i][j]
                if i == 0 and j == 0:
                    hi[0] = lo[0] = v
                    continue
                c = []
                if i > 0:
                    c += [hi[j] * v, lo[j] * v]
                if j > 0:
                    c += [hi[j - 1] * v, lo[j - 1] * v]
                hi[j], lo[j] = max(c), min(c)
        return -1 if hi[-1] < 0 else hi[-1] % MOD   # kiểm dấu TRƯỚC, modulo SAU
```

In [14]:
class Solution:
    def maxProductPath(self, grid: List[List[int]]) -> int:
        MOD = 10**9 + 7
        m, n = len(grid), len(grid[0])
        hi, lo = [0] * n, [0] * n
        for i in range(m):
            for j in range(n):
                v = grid[i][j]
                if i == 0 and j == 0:
                    hi[0] = lo[0] = v
                    continue
                c = []
                if i > 0:
                    c += [hi[j] * v, lo[j] * v]
                if j > 0:
                    c += [hi[j - 1] * v, lo[j - 1] * v]
                hi[j], lo[j] = max(c), min(c)
        return -1 if hi[-1] < 0 else hi[-1] % MOD

sol = Solution()
for g, want in examples:
    got = sol.maxProductPath(g)
    print(got, "OK" if got == want else "SAI")

-1 OK
8 OK
0 OK


## 10. Vì sao thuật toán đúng

Chứng minh quy nạp theo thứ tự duyệt (hàng rồi cột):

- **Cơ sở:** ô `(0,0)` chỉ có một đường, tích `grid[0][0]` ⇒ `hi = lo = grid[0][0]`.
- **Bước:** giả sử `hi, lo` đúng cho ô trên `U` và ô trái `L`. Tập tích các đường tới `(i,j)` là `{p·v : p ∈ P_U ∪ P_L}` với `P_U, P_L` là tập tích tới `U`, `L`. Hàm `p ↦ p·v` **đơn điệu** (tăng nếu `v ≥ 0`, giảm nếu `v < 0`, hằng nếu `v = 0`), nên max/min của tập ảnh đạt tại max hoặc min của tập gốc. Vì vậy max/min của 4 ứng viên `{hi_U·v, lo_U·v, hi_L·v, lo_L·v}` chính là max/min trên mọi đường.
- **Kết luận:** `hi[m−1][n−1]` là tích lớn nhất thật sự (số nguyên chính xác, Python không tràn). Nếu nó âm thì mọi đường đều âm ⇒ `-1`; nếu không âm thì modulo nó là đáp án — đúng như đề: "modulo sau khi đã có tích lớn nhất".

## 11. Phân tích Time / Space

Gọi `m, n` là kích thước lưới (≤ 15).

| Cách | Time | Space |
|---|---|---|
| 1. DFS mọi đường | O(C(m+n−2, m−1) · (m+n)) — hàm mũ | O(m+n) |
| 2. DP 2D | O(m·n) | O(m·n) |
| 3. DP 1D | O(m·n) | O(n) |
| 4. Memo | O(m·n) | O(m·n) |

Độ lớn số: tích tuyệt đối ≤ `4^(m+n−1) ≤ 4^29 ≈ 2.9·10¹⁷` ⇒ phép nhân số nguyên lớn trong Python vẫn coi như O(1).

In [15]:
random.seed(1)
g = [[random.randint(-4, 4) for _ in range(15)] for _ in range(15)]
for f in [max_product_dp2d, max_product_dp1d, max_product_memo]:
    t = time.perf_counter(); r = f(g)
    print(f"{f.__name__:18s} 15x15 -> {r}, {1000 * (time.perf_counter() - t):.2f} ms")

max_product_dp2d   15x15 -> 0, 0.13 ms
max_product_dp1d   15x15 -> 0, 0.16 ms
max_product_memo   15x15 -> 0, 0.27 ms


## 12. Bộ test

Ví dụ đề, edge case (1×1 âm/0/dương, một hàng, một cột, toàn 0, toàn −1, tích vượt `MOD`), và đối chiếu ngẫu nhiên với vét cạn trên lưới nhỏ.

In [16]:
ALL = [max_product_brute, max_product_dp2d, max_product_dp1d, max_product_memo,
       lambda g: Solution().maxProductPath(g)]
cases = examples + [
    ([[-3]], -1), ([[0]], 0), ([[4]], 4),
    ([[1, -1, 2, -2]], 4),                 # một hàng: chỉ 1 đường
    ([[2], [-3], [-1]], 6),                # một cột
    ([[0, 0], [0, 0]], 0),
    ([[-1, -1], [-1, -1]], -1),            # mọi đường 3 số -1 -> -1
    ([[4] * 15, [3] + [4] * 14], 4**16 % MOD),   # tích vượt MOD
]
for g, want in cases:
    for f in ALL:
        assert f(g) == want, (f, g, f(g), want)
print("Ca cố định:", len(cases), "OK")

random.seed(2024)
for _ in range(1500):
    m, n = random.randint(1, 6), random.randint(1, 6)
    g = [[random.randint(-4, 4) for _ in range(n)] for _ in range(m)]
    want = max_product_brute(g)
    for f in ALL[1:]:
        assert f(g) == want, (f, g)
print("Ngẫu nhiên nhỏ: OK")

Ca cố định: 11 OK
Ngẫu nhiên nhỏ: OK


In [17]:
# Lưới lớn (15x15): không vét cạn được, đối chiếu 4 cách DP với nhau,
# thêm lưới ít số 0 để tích lớn và nhiều đổi dấu
random.seed(99)
for _ in range(200):
    g = [[random.choice([-4, -3, -2, 2, 3, 4, 1, -1]) for _ in range(15)] for _ in range(15)]
    r = {f(g) for f in ALL[1:]}
    assert len(r) == 1
print("15x15 ngẫu nhiên: 4 cách DP luôn đồng nhất")
# kiểm tra brute force trên lưới 2x15 (chỉ 15 đường) với giá trị lớn
for _ in range(200):
    g = [[random.choice([-4, -3, 3, 4]) for _ in range(15)] for _ in range(2)]
    assert max_product_dp1d(g) == max_product_brute(g)
print("2x15 tích lớn: khớp vét cạn")

15x15 ngẫu nhiên: 4 cách DP luôn đồng nhất
2x15 tích lớn: khớp vét cạn


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ giữ max (như bài tổng)

In [18]:
def wrong_max_only(grid):
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if i == 0 and j == 0:
                dp[0][0] = grid[0][0]; continue
            c = []
            if i: c.append(dp[i - 1][j] * grid[i][j])
            if j: c.append(dp[i][j - 1] * grid[i][j])
            dp[i][j] = max(c)
    return -1 if dp[-1][-1] < 0 else dp[-1][-1] % MOD

g = examples[1][0]
print("chỉ max:", wrong_max_only(g), "| đúng:", Solution().maxProductPath(g))

chỉ max: 4 | đúng: 8


**Vì sao sai:** ở ô `(2,1)` (giá trị `−4`), bản chỉ-max chỉ nhớ `4` ở ô trên ⇒ `4·(−4) = −16` thua `3·(−4) = −12`, chọn `−12`… tích âm nhất `−2` (thứ cho `+8`) đã bị vứt từ trước. Kết quả 4 thay vì 8. **Sửa:** giữ cả `(max, min)`.

### Lỗi 2 — Lấy modulo trong lúc DP

In [19]:
def wrong_mod_inside(grid):
    m, n = len(grid), len(grid[0])
    hi = [[0] * n for _ in range(m)]; lo = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            v = grid[i][j]
            if i == 0 and j == 0:
                hi[0][0] = lo[0][0] = v; continue
            c = []
            if i: c += [hi[i-1][j] * v, lo[i-1][j] * v]
            if j: c += [hi[i][j-1] * v, lo[i][j-1] * v]
            hi[i][j] = max(c) % MOD          # SAI: modulo trước khi so sánh
            lo[i][j] = min(c) % MOD
    return hi[-1][-1] % MOD

g = [[4] * 15, [3] + [4] * 14]
print("mod sớm:", wrong_mod_inside(g), "| đúng:", Solution().maxProductPath(g))
print("tích thật lớn nhất =", 4**16, "-> % MOD =", 4**16 % MOD)

mod sớm: 221225451 | đúng: 294967268
tích thật lớn nhất = 4294967296 -> % MOD = 294967268


**Vì sao sai:** trên lưới này, đường đi hết hàng trên rồi xuống có tích `4^16`, còn đường xuống ngay có tích `3·4^15`. Sau khi lấy modulo, `4^15 % MOD` trở nên **nhỏ hơn** các giá trị khác, nên ở các ô sau DP chọn nhầm nhánh. Modulo không bảo toàn thứ tự ⇒ chỉ modulo **ở cuối**. (Ngoài ra `min % MOD` biến số âm thành dương — sai hoàn toàn về dấu.)

### Lỗi 3 — Coi `0` là "không hợp lệ"

In [20]:
def wrong_zero(grid):
    best = max_product_dp2d_raw(grid)
    return best % MOD if best > 0 else -1     # SAI: 0 là không âm!

def max_product_dp2d_raw(grid):
    m, n = len(grid), len(grid[0])
    hi, lo = [0] * n, [0] * n
    for i in range(m):
        for j in range(n):
            v = grid[i][j]
            if i == 0 and j == 0:
                hi[0] = lo[0] = v; continue
            c = []
            if i: c += [hi[j] * v, lo[j] * v]
            if j: c += [hi[j - 1] * v, lo[j - 1] * v]
            hi[j], lo[j] = max(c), min(c)
    return hi[-1]

g = examples[2][0]
print("coi 0 là xấu:", wrong_zero(g), "| đúng:", Solution().maxProductPath(g))

coi 0 là xấu: -1 | đúng: 0


**Vì sao sai:** đề yêu cầu tích **không âm** lớn nhất; `0` hợp lệ. Ví dụ 3 phải ra `0`. **Sửa:** điều kiện `best < 0` ⇒ `-1`.

### Lỗi 4 — Modulo trước khi kiểm dấu

In [21]:
def wrong_mod_before_sign(grid):
    best = max_product_dp2d_raw(grid) % MOD   # SAI: số âm % MOD thành dương
    return -1 if best < 0 else best

g = examples[0][0]
print("mod trước:", wrong_mod_before_sign(g), "| đúng:", Solution().maxProductPath(g))

mod trước: 999999971 | đúng: -1


**Vì sao sai:** trong Python `(-x) % MOD` là số dương, nên điều kiện `< 0` không bao giờ đúng ⇒ trả về một số "rác" thay vì `-1`. **Sửa:** kiểm dấu trên giá trị thật, rồi mới modulo.

### Lỗi 5 — Khởi tạo hàng/cột đầu bằng 0 hoặc quên nhân dồn

Nếu dùng chung công thức 4 ứng viên cho hàng đầu mà "ô trên" là 0 giả, tích bị ép về 0 ⇒ sai trên lưới không có số 0.

In [22]:
def wrong_zero_border(grid):
    m, n = len(grid), len(grid[0])
    hi = [[0] * n for _ in range(m)]; lo = [[0] * n for _ in range(m)]
    hi[0][0] = lo[0][0] = grid[0][0]
    for i in range(m):
        for j in range(n):
            if i == 0 and j == 0: continue
            v = grid[i][j]
            up_hi = hi[i-1][j] if i else 0; up_lo = lo[i-1][j] if i else 0     # SAI: 0 giả
            lf_hi = hi[i][j-1] if j else 0; lf_lo = lo[i][j-1] if j else 0
            c = [up_hi * v, up_lo * v, lf_hi * v, lf_lo * v]
            hi[i][j], lo[i][j] = max(c), min(c)
    return -1 if hi[-1][-1] < 0 else hi[-1][-1] % MOD

g = examples[0][0]
print("biên 0 giả:", wrong_zero_border(g), "| đúng:", Solution().maxProductPath(g))

biên 0 giả: 0 | đúng: -1


**Vì sao sai:** ô ngoài lưới **không tồn tại**, không phải "tích bằng 0". Ứng viên `0·v = 0` giả khiến ví dụ 1 trả `0` thay vì `-1`. **Sửa:** chỉ thêm ứng viên từ hướng thực sự có (`if i > 0`, `if j > 0`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Tích lớn nhất" + có số âm** ⇒ luôn giữ **cả max và min** (giống LC 152 Maximum Product Subarray).
- Lưới, chỉ đi phải/xuống ⇒ DP theo thứ tự hàng–cột, chỉ phụ thuộc ô trên và ô trái ⇒ có thể lăn mảng 1 chiều.
- **"Return modulo… after"** ⇒ giữ giá trị thật (Python) / dùng `long long` (C++), chỉ modulo ở bước cuối.
- Gặp yêu cầu "không âm"/"−1 nếu không có": kiểm dấu **trước** modulo, nhớ `0` là không âm.
- Ràng buộc nhỏ (`15 × 15`) không có nghĩa vét cạn được: `C(28,14)` > 40 triệu.

## 15. Biến thể của bài

1. **LC 152 — Maximum Product Subarray:** tích lớn nhất của đoạn con liên tiếp (giữ max/min kết thúc tại mỗi vị trí).
2. **LC 64 — Minimum Path Sum:** tổng nhỏ nhất (chỉ cần một giá trị mỗi ô).
3. **Tích nhỏ nhất** trên đường đi (đối xứng: lấy `lo` ở cuối).
4. **Đếm số đường có tích không âm** — DP đếm số đường dương / âm / bằng 0.
5. **Cho phép đi 3 hướng** (phải, xuống, chéo xuống-phải) — thêm ứng viên từ ô chéo.
6. **Có ô chướng ngại** — bỏ qua ô cấm (không có ứng viên từ đó).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS mọi đường | thử hết | hàm mũ | O(m+n) | lưới rất nhỏ, làm chuẩn đối chiếu |
| 2. DP 2D (max, min) | 4 ứng viên mỗi ô | O(mn) | O(mn) | dễ hiểu, dễ trace |
| 3. DP 1D | lăn mảng theo hàng | O(mn) | O(n) | **nộp bài**, tiết kiệm bộ nhớ |
| 4. Memo top-down | hỏi ô trên/trái | O(mn) | O(mn) | thích viết đệ quy |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[[2,-1],[3,-2]]`; (b) `[[-1]]`; (c) `[[1,-1],[-1,1]]`; (d) `[[0,-1],[-1,-1]]`.

**Bài 2 (tính tay):** Lập bảng `(hi, lo)` cho `[[1,-2],[-3,-4]]` và cho biết đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `min_product_path(grid)` trả về tích **nhỏ nhất** (không modulo) trên mọi đường.

**Bài 4 (tự viết lại):** Viết `best_path(grid)` trả về cặp `(tích lớn nhất thật, chuỗi hướng đi gồm 'R'/'D')`. Khi có nhiều đường cùng tích lớn nhất, trả đường bất kỳ.

**Bài 5 (mini challenge):** Viết `count_nonneg_paths(grid)` đếm số đường đi có tích **không âm** (DP đếm theo dấu).

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def min_product_path(grid):
    m, n = len(grid), len(grid[0])
    hi, lo = [0] * n, [0] * n
    for i in range(m):
        for j in range(n):
            # TODO: giống DP 1D, nhưng cuối cùng trả lo thay vì hi
            pass
    return None

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def best_path(grid):
    # TODO: DP (hi, lo) kèm truy vết: mỗi ô nhớ ứng viên max/min đến từ đâu và từ hi hay lo
    pass

def count_nonneg_paths(grid):
    # TODO: pos[i][j], neg[i][j], zero[i][j] = số đường có tích dương / âm / bằng 0
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: lưới 2×2 chỉ có 2 đường. Mức 2: liệt kê tích từng đường. Mức 3: (a) `2·3·(−2) = −12`, `2·(−1)·(−2) = 4` ⇒ 4; (b) −1; (c) mỗi đường có đúng một −1 ⇒ −1; (d) đều có 0 ⇒ 0.

**Bài 2.** Mức 1: hàng đầu `(1,1), (−2,−2)`; cột đầu `(−3,−3)`. Mức 2: ô `(1,1)` (giá trị −4) có ứng viên `(−2)·(−4) = 8` và `(−3)·(−4) = 12`. Mức 3: `(hi, lo) = (12, 8)`, đáp án 12.

**Bài 3.** Mức 1: DP y hệt. Mức 2: không cần kiểm dấu, không modulo. Mức 3: `return lo[-1]`.

**Bài 4.** Mức 1: lưu `src_hi[i][j] = (ô, 'hi' hoặc 'lo')`. Mức 2: truy vết ngược từ đích, chuyển giữa bảng hi/lo theo nguồn. Mức 3: nhớ đảo chuỗi hướng sau khi truy vết.

**Bài 5.** Mức 1: số đường dương mới = (dương cũ nếu v>0) hoặc (âm cũ nếu v<0). Mức 2: `v = 0` ⇒ mọi đường thành "bằng 0". Mức 3: kết quả `pos + zero` ở ô đích.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
b1 = [Solution().maxProductPath(g) for g in ([[2, -1], [3, -2]], [[-1]], [[1, -1], [-1, 1]], [[0, -1], [-1, -1]])]
print("Bài 1:", b1)
assert b1 == [4, -1, -1, 0]
assert trace_dp([[1, -2], [-3, -4]]) == 12

Bài 1: [4, -1, -1, 0]
(0,0) v= 1: khởi tạo -> hi=1, lo=1
(0,1) v=-2: trái(1,1)                ứng viên [-2, -2] -> hi=-2, lo=-2
(1,0) v=-3: trên(1,1)                ứng viên [-3, -3] -> hi=-3, lo=-3
(1,1) v=-4: trên(-2,-2) trái(-3,-3)  ứng viên [8, 8, 12, 12] -> hi=12, lo=8
bảng (hi, lo):
   [(1, 1), (-2, -2)]
   [(-3, -3), (12, 8)]
=> kết quả 12


In [26]:
# Bài 3
def min_product_path_sol(grid):
    m, n = len(grid), len(grid[0])
    hi, lo = [0] * n, [0] * n
    for i in range(m):
        for j in range(n):
            v = grid[i][j]
            if i == 0 and j == 0:
                hi[0] = lo[0] = v; continue
            c = []
            if i: c += [hi[j] * v, lo[j] * v]
            if j: c += [hi[j - 1] * v, lo[j - 1] * v]
            hi[j], lo[j] = max(c), min(c)
    return lo[-1]

random.seed(3)
for _ in range(500):
    g = [[random.randint(-4, 4) for _ in range(random.randint(1, 5))]]
    g += [[random.randint(-4, 4) for _ in range(len(g[0]))] for _ in range(random.randint(0, 4))]
    assert min_product_path_sol(g) == min(p for _, p in all_paths(g))
print("Bài 3: OK")

Bài 3: OK


In [27]:
# Bài 4
def best_path_sol(grid):
    m, n = len(grid), len(grid[0])
    # val[t][i][j]: t=0 là max, t=1 là min; src[t][i][j] = (pi, pj, pt)
    val = [[[None] * n for _ in range(m)] for _ in range(2)]
    src = [[[None] * n for _ in range(m)] for _ in range(2)]
    for i in range(m):
        for j in range(n):
            v = grid[i][j]
            if i == 0 and j == 0:
                val[0][0][0] = val[1][0][0] = v; continue
            cand = []
            for pi, pj in ((i - 1, j), (i, j - 1)):
                if pi >= 0 and pj >= 0:
                    for t in (0, 1):
                        cand.append((val[t][pi][pj] * v, (pi, pj, t)))
            mx = max(cand, key=lambda x: x[0]); mn = min(cand, key=lambda x: x[0])
            val[0][i][j], src[0][i][j] = mx
            val[1][i][j], src[1][i][j] = mn
    i, j, t = m - 1, n - 1, 0
    moves = []
    while (i, j) != (0, 0):
        pi, pj, pt = src[t][i][j]
        moves.append('D' if pi == i - 1 else 'R')
        i, j, t = pi, pj, pt
    return val[0][m - 1][n - 1], "".join(reversed(moves))

def product_of_moves(grid, moves):
    i = j = 0; p = grid[0][0]
    for c in moves:
        if c == 'D': i += 1
        else: j += 1
        p *= grid[i][j]
    assert (i, j) == (len(grid) - 1, len(grid[0]) - 1)
    return p

print("Bài 4 ví dụ 2:", best_path_sol(examples[1][0]))
random.seed(4)
for _ in range(500):
    m, n = random.randint(1, 5), random.randint(1, 5)
    g = [[random.randint(-4, 4) for _ in range(n)] for _ in range(m)]
    best, mv = best_path_sol(g)
    assert best == max(p for _, p in all_paths(g)) == product_of_moves(g, mv)
print("Bài 4: OK")

Bài 4 ví dụ 2: (8, 'DRDR')
Bài 4: OK


In [28]:
# Bài 5
def count_nonneg_paths_sol(grid):
    m, n = len(grid), len(grid[0])
    pos = [[0] * n for _ in range(m)]; neg = [[0] * n for _ in range(m)]; zer = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            v = grid[i][j]
            if i == 0 and j == 0:
                P, N, Z = (1, 0, 0) if v > 0 else (0, 1, 0) if v < 0 else (0, 0, 1)
            else:
                P = N = Z = 0
                for pi, pj in ((i - 1, j), (i, j - 1)):
                    if pi >= 0 and pj >= 0:
                        P += pos[pi][pj]; N += neg[pi][pj]; Z += zer[pi][pj]
                if v == 0:
                    P, N, Z = 0, 0, P + N + Z
                elif v < 0:
                    P, N = N, P
            pos[i][j], neg[i][j], zer[i][j] = P, N, Z
    return pos[-1][-1] + zer[-1][-1]

random.seed(5)
for _ in range(500):
    m, n = random.randint(1, 5), random.randint(1, 5)
    g = [[random.randint(-4, 4) for _ in range(n)] for _ in range(m)]
    assert count_nonneg_paths_sol(g) == sum(1 for _, p in all_paths(g) if p >= 0)
print("Bài 5: OK; ví dụ 2 có", count_nonneg_paths_sol(examples[1][0]), "đường không âm")

Bài 5: OK; ví dụ 2 có 2 đường không âm


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DP tích trên lưới:**

```python
hi, lo = [0] * n, [0] * n
for i in range(m):
    for j in range(n):
        v = grid[i][j]
        if i == 0 and j == 0: hi[0] = lo[0] = v; continue
        c = []
        if i: c += [hi[j] * v, lo[j] * v]        # ô trên
        if j: c += [hi[j-1] * v, lo[j-1] * v]    # ô trái
        hi[j], lo[j] = max(c), min(c)
return -1 if hi[-1] < 0 else hi[-1] % MOD
```

**Checklist:**
- [ ] Giữ cả **max và min** mỗi ô?
- [ ] Không thêm ứng viên "ảo" từ ngoài lưới (không dùng 0 giả)?
- [ ] **Không** modulo trong lúc DP?
- [ ] Kiểm dấu trên giá trị thật, rồi mới `% MOD`?
- [ ] `0` được coi là hợp lệ (không âm)?
- [ ] Đã thử: 1×1 âm / 0, một hàng, một cột, tích vượt `10⁹ + 7`?